# **ThermoModel v2: Bands That Account for Diminishing Returns**

The goal is unchanged: a corridor whose upper edge marks cycle tops and whose lower edge marks cycle bottoms. What changes is how the bands are built, because the original model has two structural problems that show up exactly where it hurts - predicting tops.

## What was wrong

The original ThermoModel fits a **cubic polynomial in day index** to log price, then adds **fixed offsets in log space** for the bands.

1. A cubic in raw time extrapolates explosively upward. That is the *opposite* of diminishing returns, and it is why the upper band runs away: run out-of-sample on the 2020-2025 cycle, the original approach projected a top near **$495,000** against an actual **$125,000**.
2. A fixed offset in log space means the band keeps a constant multiplicative width forever. If cycle amplitude is compressing, no fixed offset can track it - too wide late, or too narrow early. In 2017 the same approach put the upper band *below* the actual top (-39%), so price simply broke through it.

## The trend: a power law whose exponent eases off

The backbone is a log-log trend, with `L = log10(days since genesis)`:

    log10(price) = a + b·L + c·L²

A plain power law (`c = 0`) has one fixed exponent. Refitted in real time, that exponent has kept falling - 6.8 at the 2015 bottom, 5.9 at 2018, 5.8 at 2022, 5.6 today - so every fit projected the next cycle too high: forecast from the previous bottom, it overshot the 2018, 2022 and 2026 bottoms by +130%, +44% and +30%. The quadratic term builds that decline in: the exponent eases from about 5.8 in 2014 to 5.4 in 2018 and 4.9 today (4.4 by 2045; it never reaches zero on any horizon that matters). The trend grows ~31% a year today and ~25% by 2030, against 36% and 30% for the fixed exponent.

On the two cycles with enough history behind the fit, the concave trend forecast the next bottom four years ahead at -7% and +9%, and - through v2's upper band - the next top at -1% and +13%, where the fixed exponent overshot by +22% and +44% (see `backtest_trend()` and the validation below). It does need that history: fitted before 2018, on under eight years of data, it is unstable. In-sample the two fits are all but identical (R² 0.963 vs 0.962). `load_dataframe(concave_trend=False)` restores the plain power law.

## What the data actually says

Measured against that trend, the cycle extremes are:

| Cycle top | multiple of trend | | Cycle bottom | multiple of trend |
|---|---|---|---|---|
| 2011-06 | 18.4x | | 2011-11 | 0.46x |
| 2013-12 | 9.7x | | 2015-01 | 0.46x |
| 2017-12 | 5.7x | | 2018-12 | 0.54x |
| 2021-11 | 3.0x | | 2022-11 | 0.47x |
| 2025-10 | 1.4x | | 2026-07 | 0.56x |

**Tops compress hard.** The bottom multiples above look flat, but they are measured on the day of the bottom *in price*, which is not the day price sat lowest *against the trend*. Measured that way - the deepest close of each cycle - the floor tightens too: **0.32x, 0.33x, 0.42x, 0.47x, 0.55x**.

That distinction was a real bug. A lower band set at the mean of the five bottom multiples (≈ 0.50x) is a central estimate of where a bottom lands, not a floor, and it did not hold: **913 closes, 15% of the history, fell below it, the deepest 36% below**. Most of them were nowhere near a bottom - they are the long flat stretches of 2012 and 2015-16, where price idled while the trend climbed past it.

## The v2 construction

    upper(t) = trend(t) + max(a + b·cycle_position(t), upper_floor)
    lower(t) = trend(t) + min(c + d·cycle_position(t), lower_cap)

Both offsets are linear in cycle position. The upper one decays, fitted on the five observed tops. The lower one rises, fitted by **quantile regression of the daily closes against the trend** at `LOWER_TAU = 0.005` - a support rather than an average: **99.5% of the history closes above it and the deepest breach is 3%** instead of 36%, with the last two cycle bottoms landing on the band itself.

The floor stops the decaying upper band from extrapolating *below* the trend, which would be nonsense. At its default of 0 it binds on no historical cycle, so it costs nothing in backtest. The cap is its mirror: `"last"`, the default, freezes the lower band where it stands on the last day of actual data, so the tightening is read off history but never extrapolated - a rising floor under a decaying ceiling would otherwise squeeze the corridor shut.

## Validation

Leave-future-out, on the task the model exists for: for each cycle top, refit the trend and the decay on data and cycles strictly *before* it, then compare the predicted upper band against the price actually reached. The original fixed-offset approach, and v2 on a fixed-exponent power law, go through the identical test as baselines.

**Result: median absolute error 13% (v2) vs 22% (v2 on a fixed-exponent power law) vs 83% (original).** On the 2021 and 2025 tops v2 lands at -1% and +13%, where the fixed exponent overshot by +22% and +44% and the original by +83% and ~4x. The one large miss is 2017 (-79%): refitted on data up to mid-2017, only seven years of it, the concave trend bends too early - the same short-history instability the bottom backtest shows.

## The honest limitation

The fitted decay is steep enough that the upper band settles onto its asymptote - close to the trend - from around early 2028. The decay bends into that asymptote smoothly (a softplus rather than a hard clamp, which would put a visible kink in the band on the day the two met and a dead flat line after it). That is the faithful reading of five data points, but five is not many, and "future tops sit near the trend line" is an aggressive claim. `upper_floor` is exposed so the band can be given a positive asymptote instead (0.1 → tops at least 1.26x trend, 0.2 → 1.58x). Raising it changes no historical band, only the extrapolation.

*Not financial advice.*


In [ ]:
import sys
import sys, os
_base = os.getcwd()
while _base and not os.path.isdir(os.path.join(_base, 'functions')):
    _parent = os.path.dirname(_base)
    if _parent == _base: break
    _base = _parent
if os.path.abspath(_base) not in sys.path:
    sys.path.insert(0, os.path.abspath(_base))
from functions.functions_ThermoModel2 import (
    load_dataframe,
    plot_thermomodel,
    validate_top_prediction,
    backtest_trend,
    validate_lower_band,
    project_next_top,
)
import warnings
warnings.filterwarnings("ignore")

df, params = load_dataframe()

## The bands

In [ ]:
plot_thermomodel(df, params)

## Does the lower band hold?

The old flat band and the fitted support, measured against every close in the history: how many fell through, how far the deepest one went, and where each cycle bottom ends up in the corridor once the band is in place.

In [ ]:
lower_band = validate_lower_band(df, params)

## Validation: out-of-sample top prediction, v2 vs the original

In [ ]:
validation = validate_top_prediction(df)

## Why the trend is concave: every bottom, forecast in real time

Each cycle bottom forecast with only the data available at the time, from the previous bottom (about four years ahead, where the model stands today) and from the top before it (about one year ahead). The bottom is forecast as the trend times the mean bottom multiple so far - the corridor's central estimate of a bottom. Since the lower band became a support it sits below that line; what is tested here is the trend underneath both.

In [ ]:
trend_backtest = backtest_trend()

## Where does this put the next cycle top?

In [ ]:
project_next_top(df, params)

## A less aggressive assumption

The same model with a positive asymptote on the upper band, so that future tops are not assumed to converge all the way onto the trend line. Historical bands are identical; only the extrapolation moves.

In [ ]:
df_soft, params_soft = load_dataframe(upper_floor=0.15)
project_next_top(df_soft, params_soft)

## Timing

The bands say *how high and how low*, not *when*. The ThermoClock, which combines them with the cycle timing from `2a.Cycles` into a dated price path, lives in its own notebook: `1j.ThermoClock`.